# Lab — The design clinic — several good answers, and some wrong ones

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. Instructor solutions are not distributed in this repository.

**Covers.** Chapter 11 — §11.3–§11.4 (feature families; features that separate sleep stages), §11.7 (selection and the curse of dimensionality) — drawing on Chapter 4 §4.4–§4.5 (window main-lobe width and the resolution↔leakage trade) and Chapter 7 §7.5 (Welch: resolution traded against variance). A **synthesis exercise, positioned after Chapter 11**, not tied to any single chapter's lecture.

**You *design* the feature stage here and defend it — you do not pick a "correct" one.**

**Biomedical question.** Which feature design should a wearable sleep stager use — and can you *defend* it against a reasonable alternative *and* recognise the choices that would be simply wrong?
**Task type (§1.8).** Classification — feature design under a real constraint.
**Information that must be preserved.** the alpha-vs-spindle distinction that separates W from N2 (energy near 10 Hz vs 12–13 Hz); the *claim* that performance is for a new subject.
**Main assumptions.** epochs from one subject share that subject's physiology, so subjects stay whole across the split (GroupKFold).
**Primary diagnostic.** subject-independent Cohen's kappa, plus the resolution ↔ variance trade-off curve.
**Transfer challenge.** re-run the whole design for a 5 s epoch: measure what moves, what does not, and what it costs — and show the *wrong* choices stay wrong.

*This lab makes the book's worldview runnable. Two lessons sit side by side: **(1) there is
usually no single best method** — several designs are defensible and the spec picks the winner;
but **(2) that does not mean anything goes** — some choices cannot see what distinguishes the
classes, and they make the solution wrong no matter how carefully you validate. The arc is
**Spec → Design → Evaluate → Motivate → Alternatives**, and the reasoning is the point.
Self-contained: a seeded synthetic cohort, `numpy` / `scipy` / `scikit-learn` only, no data files
and no network, so it runs fully offline in a browser.*


### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab_design_clinic_no_single_best/lab_design_clinic.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab_design_clinic.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab_design_clinic.ipynb)

In [ ]:
# --- shared setup (reproducible; offline synthetic cohort) ---
import numpy as np, matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.integrate import trapezoid          # numpy has np.trapz (<2.0) OR np.trapezoid (>=2.0);
                                               # scipy's spelling works on every pinned version
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.metrics import cohen_kappa_score
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

FS = 100.0
FREQ_JITTER = 0.8   # Hz: epoch-to-epoch drift of the alpha/spindle centre frequency
STAGES = ["W", "N2", "N3"]

def synth_epoch(stage, secs, subj_offset, rng, noise=1.4):
    """The hard part is W vs N2: both have weak delta and the SAME total 8-13 Hz power, differing
    ONLY in where it sits — a ~10 Hz alpha (W) versus a ~12 Hz sigma/spindle (N2), ~2 Hz apart.
    A single wide alpha band (8-13 Hz) lumps them together and cannot tell them apart; a design
    that separates alpha (8-11) from sigma (11-16) can. N3 is delta-heavy and easy.
    The alpha/spindle centre frequency VARIES epoch to epoch (FREQ_JITTER Hz, as real rhythms do),
    so the two distributions overlap near 11 Hz — even the correct band-split design makes some
    errors (a realistic strong result, not a synthetic-perfect kappa=1)."""
    n = int(FS * secs); t = np.arange(n) / FS
    x = noise * rng.standard_normal(n) + subj_offset
    if stage == "N3":
        x += 1.6 * np.sin(2 * np.pi * 2.0 * t)                                  # strong delta
    elif stage == "N2":
        f_sigma = 12.0 + FREQ_JITTER * rng.standard_normal()                   # spindle freq varies
        x += 0.4 * np.sin(2 * np.pi * 2.0 * t)
        x += 1.0 * np.sin(2 * np.pi * f_sigma * t + rng.uniform(0, 2*np.pi))    # sigma ~12 Hz
    else:  # W
        f_alpha = 10.0 + FREQ_JITTER * rng.standard_normal()                   # alpha freq varies
        x += 0.4 * np.sin(2 * np.pi * 2.0 * t)
        x += 1.0 * np.sin(2 * np.pi * f_alpha * t + rng.uniform(0, 2*np.pi))    # alpha ~10 Hz
    return x

def build_cohort(n_subjects=12, epochs_per_stage=30, secs=30.0, seed=2026):
    rng = np.random.default_rng(seed)
    X_raw, y, groups = [], [], []
    for s in range(n_subjects):
        off = rng.normal(0, 0.4)                                                # subject nuisance
        for k, st in enumerate(STAGES):
            for _ in range(epochs_per_stage):
                X_raw.append(synth_epoch(st, secs, off, rng)); y.append(k); groups.append(s)
    return X_raw, np.array(y), np.array(groups)

def group_kfold_kappa(F, y, groups):
    # 5-fold GroupKFold (subject-independent): whole subjects are held out. This is NOT
    # leave-one-subject-out (that would be LeaveOneGroupOut, one fold per subject); with
    # 12 subjects we use 5 folds and name the function for what it actually does.
    clf = RandomForestClassifier(n_estimators=200, random_state=0)
    return cohen_kappa_score(y, cross_val_predict(clf, F, y, cv=GroupKFold(5), groups=groups))

RAW, y, groups = build_cohort()
print(f"cohort: {len(RAW)} epochs, {len(np.unique(groups))} subjects, classes = {STAGES}")


## 1. Spec — read the requirements before choosing anything

You are building the **feature stage** of a wearable sleep stager. The spec:

- single frontal EEG channel, **30 s epochs**;
- must separate **W / N2 / N3**; N2 is defined by the **spindle/sigma band (~11–16 Hz)**, which
  sits ~2 Hz from waking **alpha (~10 Hz)** — the two must not be blurred together;
- battery-limited: **fewer, cheaper features preferred**, all else equal;
- the claim is **new-subject** generalisation.

**Checkpoint 1.** In one line each: what *information must survive* the feature stage, and what is
the *hard constraint* a good design must respect?


## 2. Design — three candidates, scored the honest way

- **Design A — wrong band choice:** 3 wide band powers (delta, theta, **alpha 8–13**) from a short
  Welch. Cheap — but its single alpha band *contains both* 10 Hz and 12 Hz, merging the one
  distinction the spec needs. This is the "wrong" branch of the lab, not a cheaper-but-fine option.
- **Design B — spindle-aware, 512-pt Welch:** 4 band powers (delta, theta, **alpha 8–11**,
  **sigma 11–16**) from a longer (512-point) Welch. Separates alpha from sigma; the longer window
  also gives finer frequency resolution.
- **Design C — spindle-aware, 128-pt Welch:** the SAME four separated bands as Design B, but
  computed from Design A's cheaper 128-point Welch window.

**Design B vs Design C is the genuine no-single-best pair in this lab** — both separate alpha from
sigma, so both see the discriminator the spec needs; they trade compute (C uses a shorter, cheaper
FFT) against frequency resolution (B's is finer, which would start to matter under a spec change —
e.g. a narrower sigma band, or an added N1 stage). Design A is different in kind: it cannot see the
discriminator at all, at any cost.

All three are scored with the **same** subject-independent (GroupKFold) harness.

**Checkpoint 2.** Which two stages does Design A confuse? Score it the same subject-independent
way, print its confusion matrix with stage labels, and name the largest off-diagonal cell.


In [ ]:
def bandpowers(x, bands, nperseg):
    # Integrated power = area under the PSD, not a bare bin sum (Ch. 7's named
    # "common student mistake"): interpolate the exact band edges onto the PSD
    # so adjacent bands share their boundary point instead of each losing the
    # half-bin trapezoid slice that straddles it.
    nper = min(nperseg, len(x))
    f, p = welch(x, fs=FS, nperseg=nper, noverlap=nper // 2)
    out = []
    for lo, hi in bands:
        lo_c, hi_c = max(lo, f[0]), min(hi, f[-1])
        inside = (f > lo_c) & (f < hi_c)
        f_band = np.concatenate(([lo_c], f[inside], [hi_c]))
        p_band = np.concatenate(([np.interp(lo_c, f, p)], p[inside], [np.interp(hi_c, f, p)]))
        out.append(trapezoid(p_band, f_band))
    return out

BANDS_A = [(0.5, 4), (4, 8), (8, 13)]                    # coarse: one alpha band holds 10 AND 12 Hz
BANDS_B = [(0.5, 4), (4, 8), (8, 11), (11, 16)]          # fine: alpha 8-11 vs sigma 11-16 (separated)

FA = np.array([bandpowers(x, BANDS_A, 128) for x in RAW])
FB = np.array([bandpowers(x, BANDS_B, 512) for x in RAW])
FC = np.array([bandpowers(x, BANDS_B, 128) for x in RAW])   # Design C: B's separated bands, A's cheap window
kA, kB = group_kfold_kappa(FA, y, groups), group_kfold_kappa(FB, y, groups)
kC = group_kfold_kappa(FC, y, groups)
print(f"Design A (3 coarse feats):                 subject-independent kappa = {kA:.2f}")
print(f"Design B (4 feats, 512-pt Welch):           subject-independent kappa = {kB:.2f}")
print(f"Design C (4 feats, 128-pt Welch, cheaper):  subject-independent kappa = {kC:.2f}")
# TODO (Checkpoint 2): add a confusion matrix for A — WHICH two stages does the coarse band confuse?
#   Score Design A the SAME subject-independent way (cross_val_predict + GroupKFold(5)), build the
#   confusion matrix, print it with the stage labels, and read off the largest off-diagonal cell.
#   Use sklearn.metrics.confusion_matrix(y, pred, labels=[0, 1, 2]) -- rows are the TRUE stage in
#   STAGES order, which is what the sanity check indexes -- and the same
#   RandomForestClassifier(n_estimators=200, random_state=0) so the matrix matches kA.
#   Bind the matrix to CM_A — the sanity check at the end uses it.
raise NotImplementedError("TODO: implement this — see the comment above")


## 3. The genuine trade-off — resolution ↔ variance (no single best segmentation)

Within a spectral design there is **no single best Welch setting**. The knob is the **segment
length `nperseg`**; from it, the signal length `L` and the overlap fix the *actual* number of
averaged segments — **`n_seg = 1 + (L - nperseg)//(nperseg - noverlap)`** — *not* a free `K` you
get to name. Longer segments give **finer frequency resolution** but leave **fewer segments to
average**, so the estimate is **noisier**; shorter segments do the reverse. The cell **computes**
effective resolution from a fixed formula (4× bin spacing) and **measures** variance directly (by
repeating the Welch estimate over independent noise draws) as it sweeps `nperseg`, and reports the
real `n_seg`.

Two labels that are easy to conflate — the cell keeps them separate:
- **bin spacing** `= FS/nperseg` — how far apart the FFT bins sit;
- **effective resolution** `≈ 4 × FS/nperseg` — how far apart two tones must be to be told apart
  (the default Hann window's main lobe is 4 bins wide, per Ch. 4's window table — not to be
  confused with its ~1.5-bin *ENBW*, a different quantity governing noise, not resolution). To
  split waking **alpha (10 Hz)** from the **spindle (12 Hz)** the *effective* resolution — not
  the spacing — must beat ~2 Hz.

Read that claim precisely: it is about **resolving two peaks on the PSD** — seeing them as two
bumps. It is *not* the same as the band-power feature in §2 being able to tell the stages apart,
because there the discrimination comes from **where the band edges sit** (8–11 vs 11–16), which
moves energy between two features even when the main lobe is too wide to show two separate peaks.
Keeping those two questions apart is part of the exercise — §5 measures the difference.

In [ ]:
# We drive the sweep by nperseg and compute the real n_seg from L, nperseg, noverlap; we keep
# bin spacing (FS/nperseg) separate from effective resolution (4x, Hann main lobe); and we probe
# a fixed physical frequency (13 Hz), mapped to the nearest bin per nperseg.
L = len(RAW[0])                       # 3000 samples = 30 s at FS = 100 Hz
npersegs = [64, 128, 256, 512, 1024, 2048]
MAINLOBE_HANN = 4.0                   # Hann main-lobe width in BINS (Ch. 4 table) -> spacing x this = effective res
SPINDLE = (11.0, 16.0)                # FIXED physical band (Hz) the design must be able to resolve
F0 = 13.0                             # probe frequency (Hz), fixed in Hz -> bin index per nperseg
rng = np.random.default_rng(7)

n_seg, spacing, eff_res, est_var = [], [], [], []
for nper in npersegs:
    nover = nper // 2
    k = 1 + (L - nper) // (nper - nover)          # ACTUAL number of Welch segments
    df = FS / nper                                # bin SPACING (Hz) — NOT the resolution
    n_seg.append(k); spacing.append(df); eff_res.append(MAINLOBE_HANN * df)
    vals = []
    for _ in range(120):                          # variance of the PSD estimate at F0 over noise
        f, S = welch(rng.standard_normal(L), fs=FS, nperseg=nper, noverlap=nover)
        vals.append(S[np.argmin(np.abs(f - F0))]) # fixed PHYSICAL freq -> nearest bin, per nperseg
    est_var.append(float(np.var(vals)))

print("nperseg  n_seg  spacing(Hz)  eff-res(Hz)  var@13Hz")
for nper, k, df, er, v in zip(npersegs, n_seg, spacing, eff_res, est_var):
    print(f"{nper:6d}  {k:5d}  {df:10.3f}  {er:10.3f}  {v:9.2e}")

order = np.argsort(n_seg)
xs      = np.array(n_seg)[order]
er_s    = np.array(eff_res)[order]
var_s   = np.array(est_var)[order]
fig, axL = plt.subplots(figsize=(6.8, 3.9))
axL.plot(xs, er_s, 'o-', color='tab:blue'); axL.axhline(2.0, color='tab:blue', ls=':', lw=1)
axL.set_xlabel('actual number of Welch segments  n_seg  (more = shorter segments)')
axL.set_ylabel('effective resolution (Hz) — lower finer', color='tab:blue')
axL.set_xscale('log', base=2); axL.set_xticks(xs); axL.set_xticklabels(xs)
axR = axL.twinx(); axR.plot(xs, var_s, 's-', color='tab:red'); axR.set_yscale('log')
axR.set_ylabel('variance of PSD estimate @13 Hz — lower smoother', color='tab:red')
plt.title('No single best segmentation: finer resolution (fewer segments) costs variance')
plt.tight_layout(); plt.show()
print('bin SPACING (FS/nperseg) is not the resolution: the Hann window blurs by 4 bins (its')
print('main-lobe width, Ch. 4), so the EFFECTIVE resolution is 4x the spacing. To split 10 vs 12 Hz')
print('it must stay < 2 Hz (dotted line) — few, long segments — while a low-variance estimate wants')
print('many segments: that is the trade-off, and no setting wins both ends of it.')

## 4. …but not everything is a trade-off — some choices are just wrong

"No single best" is **not** "anything goes." A design is wrong when it *cannot see what
distinguishes the classes*, and then no amount of careful validation rescues it. Two examples on
*this* spec (where the discriminator is *which* frequency, 10 vs 12 Hz, carries the energy):

- **Wrong band choice — Design A above.** Its single alpha band (8–13 Hz) contains both 10 and
  12 Hz, so it reports *nearly the same* number for W and N2. That is not a cheaper-but-fine
  option; it is wrong *for this spec* — it destroys the one distinction the spec says must survive.
  (Caveat: with very short segments, e.g. `nperseg = 64`, leakage across the 8 Hz edge into the
  theta feature can smuggle part of the 10-vs-12 Hz information back in — Design A then scores
  better. That is an accident of the window, not something you could defend or rely on; a design
  whose discrimination depends on leakage is still wrong.)
- **Wrong feature *type*.** Time-domain **amplitude** features (mean, std, RMS, peak-to-peak)
  measure *how much* energy there is, never *where* it sits in frequency. W and N2 have the same
  amplitude, so amplitude features cannot separate them at all.

Run it: on the W-vs-N2 pair the amplitude features sit at chance and Design A is close to it — even
though both keep a respectable 3-class kappa because N3 is easy.


In [ ]:
FT = np.array([[np.mean(x), np.std(x), np.sqrt(np.mean(x**2)), np.ptp(x)] for x in RAW])  # amplitude-only
kT = group_kfold_kappa(FT, y, groups)
print(f"RIGHT  — Design B (separated bands, 512-pt Welch):   kappa = {kB:.2f}")
print(f"RIGHT  — Design C (separated bands, 128-pt Welch):   kappa = {kC:.2f}   (matches B, cheaper)")
print(f"WRONG  — Design A (coarse alpha band):                kappa = {kA:.2f}   (blurs 10 vs 12 Hz)")
print(f"WRONG  — amplitude-only features:                     kappa = {kT:.2f}   (blind to frequency)")

# A 3-class kappa can hide failure on the hard pair because N3 is easy, so compute W-vs-N2
# directly: restrict to W (0) and N2 (1), score the SAME subject-independent way.
# Chance ~ kappa 0 / accuracy 0.5.
from sklearn.metrics import accuracy_score
def wvsN2(F):
    m = (y == 0) | (y == 1)
    clf = RandomForestClassifier(n_estimators=200, random_state=0)
    pred = cross_val_predict(clf, F[m], y[m], cv=GroupKFold(5), groups=groups[m])
    return cohen_kappa_score(y[m], pred), accuracy_score(y[m], pred)
print("\nW-vs-N2 only (the one distinction the spec insists must survive):")
for name, F in [("Design B (right)", FB), ("Design C (right)", FC), ("Design A (coarse)", FA), ("amplitude-only", FT)]:
    k, a = wvsN2(F)
    print(f"   {name:18s} kappa = {k:5.2f}   accuracy = {a:4.2f}")
print("The amplitude-only design sits at chance (kappa≈0, accuracy≈0.5); the coarse-band design")
print("does little better on W-vs-N2 (kappa≈0.2) — both fall far short of the band-split designs")
print("(B and C, which agree with each other despite C's cheaper window).")
print("A decent-looking 3-class kappa (Design A ≈0.6) can hide this failure on the hard pair.")

## 5. Motivate & Alternatives — defend a choice, argue against yourself, and mark the boundary

**Checkpoint 3 (the heart of the lab).**
1. **Motivate:** which spectral design do you ship *for this spec*, and why — tie it to the kappa
   gap *and* the compute preference (how much kappa per extra feature/FFT length?).
2. **Alternatives (defensible):** name a spectral design or Welch segment length `nperseg` you did
   **not** pick and a concrete **spec change** that would make it win — e.g., a 5 s epoch, an added
   N1 stage, or an MCU that forbids a 512-point FFT.
3. **Boundary (wrong):** state, in one sentence each, *why* the coarse-band and amplitude-only
   choices are wrong here — and what feature of the **spec** makes them wrong (not a matter of taste).
4. **Transfer:** in the workspace cell below, build a SECOND cohort with `build_cohort(secs=5.0)`
   and rescore all three designs — do not modify §2, the sanity check needs both. Does the
   *defensible* B/C winner move? Do the *wrong* choices become right? Explain both using the
   resolution ↔ variance curve.


In [ ]:
# --- your workspace: run the 5 s transfer test and re-evaluate ---
# TODO: build a SECOND cohort with secs=5.0 (do not modify §2 — the sanity check needs both the
#   30 s and 5 s numbers), recompute the three feature sets (A / B / amplitude-only) and score them
#   with the SAME harness. Report, for each: the 3-class kappa and the W-vs-N2 kappa, at 30 s and
#   at 5 s side by side. Also report the ACTUAL number of averaged Welch segments (n_seg) at 5 s.
#   NOTE: bandpowers() clamps nperseg to the epoch length, so at 5 s Design B uses nperseg = 500,
#   not 512 — apply the n_seg formula with the clamped value (you should get n_seg = 1 for B and
#   6 for A). Predict first, then run, then explain what moved and what did not.
#
#   The sanity check at the end reads these names, so bind exactly them:
#       RAW5, y5, groups5   <- the 5 s cohort from build_cohort(secs=5.0)
#       FA5, FB5, FT5       <- the three 5 s feature matrices (coarse / band-split / amplitude-only)
#       kA5, kB5, kT5       <- their 3-class subject-independent kappas
#       wvsN2_at(F, yy, gg) <- returns ONLY the W-vs-N2 kappa as a float (NOT the (kappa, accuracy)
#                              tuple that §4's wvsN2() returns — the sanity check calls abs() on
#                              it), for ANY cohort, so the same binary check can be run at 30 s and
#                              at 5 s (the wvsN2() defined in section 4 is hard-wired to the 30 s
#                              y/groups and cannot be reused)
raise NotImplementedError("TODO: implement this — see the comment above")


### Live sanity check
A design claim you never verify is an opinion. These asserts run on the numbers computed above and
encode the clinic's three claims: the band-split design really does beat the coarse one on the
subject-independent harness, at both epoch lengths; Design A's errors really are concentrated on
**W↔N2** and not spread across all three stages; and the amplitude-only design really sits at
chance on W-vs-N2 at both epoch lengths, while Design A's errors are heavily concentrated on that
same pair rather than Design A itself being at chance.

In [ ]:
# --- live sanity check: every number below was computed by the cells above ---
# (1) the defensible design wins on the subject-independent harness, at both epoch lengths.
assert kB > kA + 0.15, f"Design B should clearly beat Design A, got {kB:.2f} vs {kA:.2f}"
assert kB5 > kA5 + 0.15, f"...and still at 5 s, got {kB5:.2f} vs {kA5:.2f}"

# (2) Design A does not fail everywhere -- it fails on EXACTLY the W<->N2 distinction.
assert CM_A[2, 2] / CM_A[2].sum() > 0.95, "N3 must stay easy for Design A (it is delta-heavy)"
wn2_err = (CM_A[0, 1] + CM_A[1, 0]) / CM_A[:2, :2].sum()
assert wn2_err > 0.25, f"Design A must confuse W and N2 heavily, got {wn2_err:.2f}"

# (3) the WRONG choices are wrong, not merely worse -- at chance on W vs N2, at both lengths.
k_amp_30 = wvsN2_at(FT, y, groups)
k_amp_5 = wvsN2_at(FT5, y5, groups5)
assert abs(k_amp_30) < 0.10 and abs(k_amp_5) < 0.10, \
    f"amplitude-only must sit at chance on W-vs-N2: {k_amp_30:.2f} / {k_amp_5:.2f}"
assert wvsN2_at(FB, y, groups) > 0.60, "the band-split design must genuinely separate W from N2"
assert kT5 < kB5 - 0.30, "amplitude-only must stay far behind the defensible design at 5 s"

print("sanity check PASSED:")
print(f"  3-class kappa   B {kB:.2f} -> {kB5:.2f}   A {kA:.2f} -> {kA5:.2f}   amp {kT:.2f} -> {kT5:.2f}  (30 s -> 5 s)")
print(f"  Design A errors concentrate on W<->N2 ({wn2_err:.0%} of W/N2 epochs) while N3 stays clean")
print(f"  amplitude-only W-vs-N2 kappa {k_amp_30:.2f} (30 s) / {k_amp_5:.2f} (5 s): chance, both times")

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the
**defense and the boundary**, not which design you chose.

1. *Spec:* what had to survive the feature stage, and what was the hard constraint?
2. *Design & evaluate:* which spectral design did you ship, and what did the numbers say?
3. *Motivate:* justify it against the trade-off — be specific (kappa per extra feature).
4. *Alternatives:* the *defensible* design you rejected, and the spec change that would make it win.
5. *Boundary:* the two *wrong* choices, and the exact spec feature that makes each one wrong.
6. *Transfer:* what moved at 5 s and what did not — and what did the shorter epoch **cost**?
   Say explicitly which of your predictions the measurement refuted, and what that taught you about
   which question the resolution↔variance curve actually answers.

> *Your answers here.*

**Further practice — the validation axis.** This lab put the trade-off in the *feature* stage. The
same "several good choices, but also wrong ones" logic governs the *validation* stage: plain k-fold
is a wrong choice when subjects, sites, or time create dependencies. Try the 20-scenario quiz and the
worked clinical demonstrations at `ki-smile.github.io/trustcv`, and use TrustCV's `DataLeakageChecker`
on your own split.

---
*Design-clinic lab for **Biomedical Signal Processing & Data Analytics**. Synthetic cohort;
illustrative numbers. Two lessons: there is rarely a single best design (defend your optimum and
name the alternative), **and** some choices are simply wrong (they cannot see what the spec needs).*
